In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Preliminaries

In [1]:
!pip install -q transformers

In [2]:
import numpy as np
import pandas as pd
from sklearn import metrics
import transformers
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertTokenizer, BertModel, BertConfig

In [3]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

### Dataset

In [6]:
import pandas as pd
result = pd.read_csv('/content/drive/MyDrive/Modeling/BERT/final_final_result_by_Gemini_Changed_Features.csv')

In [7]:
result = result.iloc[:, 1:]

In [8]:
result

,Title,Artist,Genre,Url,Happy,Sad,Energized,Relaxed,Nostalgic,Romantic,Angry,Focused,Inspired,Melancholic,Peaceful,Anxious,Confident,Dreamy,Lonely
0,Love wins all,아이유,Ballad,https://www.youtube.com/watch?v=KBGeoGl6AVA&pp...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,비의 랩소디,임재현,Ballad,https://www.youtube.com/watch?v=TyCYGCCD6iQ&pp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,그대만 있다면 (여름날 우리 X 너드커넥션 (Nerd Connection)),너드커넥션 (Nerd Connection),Ballad,https://www.youtube.com/watch?v=IMWT6937uUs&pp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,헤어지자 말해요,박재정,Ballad,https://www.youtube.com/watch?v=SrQzxD8UFdM&pp...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,인사,범진,Ballad,https://www.youtube.com/watch?v=q0oUosezzSg&pp...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2071,Life Goes On,방탄소년단,Hiphop,https://www.youtube.com/watch?v=r27uNWGTGfk&pp...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
2072,Catch (Feat. 화사),에픽하이 (EPIK HIGH),Hiphop,https://www.youtube.com/watch?v=-yYAKF62Ujs&pp...,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2073,CASE 143,Stray Kids (스트레이 키즈),Hiphop,https://www.youtube.com/watch?v=zuYSfn-C91c&pp...,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2074,WE (Feat. 박재범) (Prod. by Slom),XINSAYNE,Hiphop,https://www.youtube.com/watch?v=vswVFvRvrRU&pp...,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


In [9]:
import ast
new_df = result[['Description', 'feature_list']]
new_df['feature_list'] = new_df['feature_list'].apply(lambda x: ast.literal_eval(x))
new_df

KeyError: "None of [Index(['Description', 'feature_list'], dtype='object')] are in the [columns]"

In [ ]:
for index in range(len(new_df)):
  if type(new_df.loc[index, 'feature_list']) != list :
    print(index)

In [ ]:
import math
for index in range(len(new_df)):
  for i in new_df.loc[index, 'feature_list']:
    if math.isnan(i):
      print(index)

In [ ]:
final_df = pd.DataFrame(columns = ['comment_text', 'list'])
final_df['comment_text'] = new_df['Description']
final_df['list'] = new_df['feature_list']
final_df

In [ ]:
new_df = final_df

### Model

In [10]:
MAX_LEN = 200
TRAIN_BATCH_SIZE = 8
VALID_BATCH_SIZE = 4
EPOCHS = 50
LEARNING_RATE = 1e-05

In [11]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [12]:
class CustomDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.comment_text = dataframe.comment_text
        self.targets = self.data.list
        self.max_len = max_len

    def __len__(self):
        return len(self.comment_text)

    def __getitem__(self, index):
        comment_text = str(self.comment_text[index])
        comment_text = " ".join(comment_text.split())

        inputs = self.tokenizer.encode_plus(
            comment_text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            pad_to_max_length=True,
            return_token_type_ids=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }

In [13]:
# Creating the dataset and dataloader for the neural network

train_size = 0.8
train_dataset=new_df.sample(frac=train_size,random_state=200)
test_dataset=new_df.drop(train_dataset.index).reset_index(drop=True)
train_dataset = train_dataset.reset_index(drop=True)


print("FULL Dataset: {}".format(new_df.shape))
print("TRAIN Dataset: {}".format(train_dataset.shape))
print("TEST Dataset: {}".format(test_dataset.shape))

training_set = CustomDataset(train_dataset, tokenizer, MAX_LEN)
testing_set = CustomDataset(test_dataset, tokenizer, MAX_LEN)

NameError: name 'new_df' is not defined

In [ ]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

In [16]:
class BERTClass(torch.nn.Module):
    def __init__(self):
        super(BERTClass, self).__init__()
        self.l1 = transformers.BertModel.from_pretrained('bert-base-uncased')
        self.l2 = torch.nn.Dropout(0.3)
        self.l3 = torch.nn.Linear(768, 15)

    def forward(self, ids, mask, token_type_ids):
        _, output_1= self.l1(ids, attention_mask = mask, token_type_ids = token_type_ids, return_dict=False)
        output_2 = self.l2(output_1)
        output = self.l3(output_2)
        return output

model = BERTClass()
model.to(device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BERTClass(
  (l1): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=Tr

In [ ]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

In [ ]:
optimizer = torch.optim.Adam(params =  model.parameters(), lr=LEARNING_RATE)

## Fine Tuning!

### Train

In [ ]:
def train(epoch):
    model.train()
    for _,data in enumerate(training_loader, 0):
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        outputs = model(ids, mask, token_type_ids)
        # print("\nOutputs")
        # print(outputs)
        # print("\nTargets")
        # print(targets)
        optimizer.zero_grad()
        # outputs = outputs.int()
        loss = loss_fn(outputs, targets)
        if _%5000==0:
            print(f'Epoch: {epoch}, Loss:  {loss.item()}')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:
for epoch in range(EPOCHS):
    train(epoch)

### Validation

In [ ]:
def validation(epoch):
    model.eval()
    fin_targets=[]
    fin_outputs=[]
    with torch.no_grad():
        for _, data in enumerate(testing_loader, 0):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            print("IDS")
            print(data['ids'].shape)
            print(data['mask'].shape)
            print(data['token_type_ids'].shape)
            break
            targets = data['targets'].to(device, dtype = torch.float)
            outputs = model(ids, mask, token_type_ids)
            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())
    return fin_outputs, fin_targets

In [ ]:
for epoch in range(5):
    outputs, targets = validation(epoch)
    outputs = np.array(outputs) >= 0.5
    accuracy = metrics.accuracy_score(targets, outputs)
    f1_score_micro = metrics.f1_score(targets, outputs, average='micro')
    f1_score_macro = metrics.f1_score(targets, outputs, average='macro')
    print(f"Accuracy Score = {accuracy}")
    print(f"F1 Score (Micro) = {f1_score_micro}")
    print(f"F1 Score (Macro) = {f1_score_macro}")
    break

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/DSL/모델링 추천시스템/BERT_model_parameters.pth')

### Features

In [ ]:
features = """
Happy
Sad
Energized
Relaxed
Nostalgic
Romantic
Angry
Focused
Inspired
Melancholic
Peaceful
Anxious
Confident
Dreamy
Lonely
"""
features_list = features.split()

### Trying on new Text

In [ ]:
# colab 환경에서 학습을 진행하실 분들은 구글드라이브를 연동해주세요
from google.colab import drive
drive.mount('/content/drive')

In [14]:
import numpy as np
import pandas as pd
dfdfdfdf = pd.read_csv('/content/drive/MyDrive/Modeling/BERT/final_final_result_by_Gemini_Changed_Features.csv')
changed_df = pd.read_csv('/content/drive/MyDrive/Modeling/BERT/with_actual_description_result_Changed_Features.csv')
our_data = changed_df.iloc[:, -15:].to_numpy()

In [ ]:
torch.save(model.state_dict(), '/content/drive/MyDrive/DSL/모델링 추천시스템/BERT_model_parameters.pth')

In [17]:
model.load_state_dict(torch.load('/content/drive/MyDrive/Modeling/BERT/BERT_model_parameters.pth'))

<All keys matched successfully>

In [124]:
#comment_text = 'When you got rejected by a girl'
#comment_text = 'When you want to go to sleep'
comment_text = 'I need to get motivated to exercise'
#comment_text = 'I need songs to listen to when going on a date'
test_df = pd.DataFrame({'comment_text' : comment_text, 'list' : [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]})


new_set = CustomDataset(test_df, tokenizer, MAX_LEN)

new_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

new_loader = DataLoader(new_set, **new_params)

fin_outputs=[]
for _,data in enumerate(new_loader, 0):
  ids = data['ids'].to(device, dtype = torch.long)
  mask = data['mask'].to(device, dtype = torch.long)
  token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
  targets = data['targets'].to(device, dtype = torch.float)

  outputs = model(ids, mask, token_type_ids)
  fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())
  real_outputs = np.array(fin_outputs) >= 0.5
  real_outputs = real_outputs[0]
  real_outputs = real_outputs.astype(int)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2645: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [125]:
real_probs = [round(i, 3) for i in fin_outputs[0]]
real_probs

[0.004,
 0.001,
 0.029,
 0.084,
 0.002,
 0.001,
 0.0,
 0.512,
 0.997,
 0.002,
 0.017,
 0.001,
 0.432,
 0.024,
 0.001]

In [71]:
import pickle
with open('/content/drive/MyDrive/Modeling/finaldata/feature_database.pkl', 'rb') as f:
  feature_matrix = pickle.load(f)

In [72]:
with open('/content/drive/MyDrive/Modeling/finaldata/labelinfo.pickle', 'rb') as f:
  labelinfo = pickle.load(f)

In [73]:
with open('/content/drive/MyDrive/Modeling/finaldata/id_to_songs.pkl', 'rb') as f:
  id_to_songs = pickle.load(f)

In [74]:
def getLabel(result, labeldict):
  result_label_pairs = [(value, labeldict[idx]) for idx, value in enumerate(result)]

  sorted_pairs = sorted(result_label_pairs, key=lambda x: x[0], reverse=True)

  pos_labels = [f"{label} : {value}" for value, label in sorted_pairs if value > 0.5]

  return pos_labels

def getSong(idx, songdict):
  print(f"Title : {songdict[idx][0]}")
  print(f"Artist : {songdict[idx][1]}")

In [121]:
import torch
import torch.nn.functional as F


def like_contrastive(query_vector, vector_dict, top_k=5, top_k_positive=2, top_k_negative=5):

    query_vector = torch.tensor(query_vector)

    # 쿼리 벡터에서 확률 값이 높은 top_k_positive개의 인덱스 선택
    _, query_positive_indices = torch.topk(query_vector, k=top_k_positive)

    # 쿼리 벡터에서 확률 값이 낮은 top_k_negative개의 인덱스 선택
    _, query_negative_indices = torch.topk(query_vector, k=top_k_negative, largest=False)

    # 선택된 인덱스에 해당하는 쿼리 벡터의 값 선택
    target_indices = list(query_positive_indices) + list(query_negative_indices)
    target_indices = [tensor.item() for tensor in target_indices]


    top_idx = {}

    query = query_vector[target_indices]
    for idx, value in vector_dict.items():
      target = torch.tensor(value)
      target = target[target_indices]

      query = query.flatten()  # Flatten the query tensor
      target = target.flatten()  # Flatten the target tensor
      similarity = torch.norm(query.unsqueeze(0) - target.unsqueeze(0))
      top_idx[idx] = similarity

    sorted_dict = sorted(top_idx.items(), key=lambda x: x[1], reverse=True)

    # 상위 top_k개의 key 선택
    top_keys = [key for key, _ in sorted_dict[:top_k]]

    return top_keys

In [104]:
def use_euclidean(query_vector, vector_dict, top_k=5):

    query_vector = torch.tensor(query_vector)

    top_idx = {}

    for idx, value in vector_dict.items():
      target = torch.tensor(value)

      query = query_vector.flatten()
      target = target.flatten()

      similarity = torch.norm(query.unsqueeze(0) - target.unsqueeze(0))
      top_idx[idx] = similarity

    sorted_dict = sorted(top_idx.items(), key=lambda x: x[1], reverse=True)

    # 상위 top_k개의 key 선택
    top_keys = [key for key, _ in sorted_dict[:top_k]]

    return top_keys

In [92]:
import torch

def use_hamming(query_vector, vector_dict, top_k=5):
    query_vector = torch.tensor(query_vector)
    top_idx = {}
    for idx, value in vector_dict.items():
        target = torch.tensor(value)
        query = query_vector.flatten().bool()  # Convert to boolean tensor
        target = target.flatten().bool()  # Convert to boolean tensor
        similarity = torch.sum(query != target)  # Calculate Hamming distance
        top_idx[idx] = similarity

    sorted_dict = sorted(top_idx.items(), key=lambda x: x[1])

    # 상위 top_k개의 key 선택
    top_keys = [key for key, _ in sorted_dict[:top_k]]

    return top_keys

In [126]:
index_list = like_contrastive(real_probs, feature_matrix)

In [95]:
index_list = use_euclidean(real_probs, feature_matrix)

In [119]:
index_list = use_hamming(real_probs, feature_matrix)

In [127]:
print(comment_text)
print(getLabel(real_probs, labelinfo))

print("----------------------------------------")
for i, index in  enumerate(index_list):
  print(f"Top {i+1}")
  getSong(index, id_to_songs)
  print("----------------------------------------")

I need to get motivated to exercise
['Inspired : 0.997', 'Focused : 0.512']
----------------------------------------
Top 1
Title : 12월의 기적 (Miracles In December)
Artist : EXO
----------------------------------------
Top 2
Title : 화이트(WHITE) (Feat. 박재범)
Artist : 다비치
----------------------------------------
Top 3
Title : MAGIC!
Artist : Zior Park
----------------------------------------
Top 4
Title : Love You More
Artist : Mount Cashmore
----------------------------------------
Top 5
Title : Xo Tour Llif3
Artist : Lil Uzi Vert
----------------------------------------


In [ ]:
for i in range(len(real_outputs)):
  if real_outputs[i] == 1:
    print(features_list[i])

In [ ]:
real_outputs.tolist()

[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0]

### Comparison

In [ ]:
from scipy.spatial.distance import cdist

dists = cdist([real_probs], our_data, metric='euclidean')
most_similar_indices = np.argsort(dists[0])[:5]
print("Indices of most similar arrays:", most_similar_indices)
print("Hamming distances:", dists[0][most_similar_indices])

Indices of most similar arrays: [ 606  861 1117  897 1104]
Hamming distances: [0.70107418 0.70107418 0.70107418 0.74933637 0.74933637]


In [ ]:
changed_df.iloc[most_similar_indices, [2, 3, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]]

,Title,Singer,Happy,Sad,Energized,Relaxed,Nostalgic,Romantic,Angry,Focused,Inspired,Melancholic,Peaceful,Anxious,Confident,Dreamy,Lonely
606,Sleep Well Beast!,The National,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
861,Sprinter,Dave & Central Cee,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
1117,goodnight n go!,Ariana Grande,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
897,305 to My City!,Drake,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0
1104,Counting Stars!,OneRepublic,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0
